# Solución · Presupuesto de una actividad

Este notebook resuelve las tres consignas de "Para expandir" del programa base:

1. Método que informa cuál es el rubro más costoso.
2. Porcentaje de imprevistos aplicado al cálculo total.
3. Un segundo objeto `Actividad` con otro presupuesto y otra cantidad de asistentes, comparado contra el primero.

La clase se reescribe para que cada mejora quede como un método propio y reutilizable (nada de cálculos sueltos por fuera del objeto), y la comparación se resuelve con una función que acepta **cualquier cantidad** de actividades, no solo dos.

## 1. Clase `Actividad` extendida

Cambios respecto del programa base:

- `calcular_total()` ahora se apoya en `calcular_subtotal()` (suma de rubros) y `calcular_imprevistos()` (porcentaje sobre el subtotal), así cada cálculo tiene un único responsable.
- `porcentaje_imprevistos` es un atributo más del objeto: cada actividad puede tener el suyo, con 0 % como valor por defecto.
- `rubro_mas_costoso()` devuelve el rubro de mayor costo usando `max()`, sin recorrer la lista "a mano".
- `generar_informe()` arma el reporte como texto reutilizable, para no repetir los `print()` cada vez que se necesita mostrar una actividad.

In [ ]:
class Actividad:
    def __init__(
        self,
        nombre: str,
        presupuesto: float,
        asistentes: int,
        porcentaje_imprevistos: float = 0.0,
    ) -> None:
        # Estos atributos guardan los datos propios de cada actividad.
        self.nombre: str = nombre
        self.presupuesto: float = presupuesto
        self.asistentes: int = asistentes
        self.porcentaje_imprevistos: float = porcentaje_imprevistos
        self.rubros: list[tuple[str, float]] = []

    def agregar_rubro(self, nombre: str, costo: float) -> None:
        # El nuevo rubro se agrega a la lista del objeto.
        self.rubros.append((nombre, costo))

    def calcular_subtotal(self) -> float:
        # Suma de los rubros cargados, sin imprevistos.
        total: float = 0.0
        for _, costo in self.rubros:
            total += costo
        return total

    def calcular_imprevistos(self) -> float:
        # Porcentaje de imprevistos aplicado sobre el subtotal.
        return self.calcular_subtotal() * (self.porcentaje_imprevistos / 100)

    def calcular_total(self) -> float:
        # Subtotal + imprevistos: este es el costo real a cubrir.
        return self.calcular_subtotal() + self.calcular_imprevistos()

    def calcular_costo_por_persona(self) -> float:
        if self.asistentes <= 0:
            return 0.0
        return self.calcular_total() / self.asistentes

    def evaluar_presupuesto(self) -> str:
        diferencia: float = self.presupuesto - self.calcular_total()

        if diferencia >= 0:
            return f"Viable: quedan ${diferencia:,.2f} disponibles"

        return f"No viable: faltan ${abs(diferencia):,.2f}"

    def rubro_mas_costoso(self) -> tuple[str, float] | None:
        # None si todavía no se cargó ningún rubro: evita romper con lista vacía.
        if not self.rubros:
            return None
        return max(self.rubros, key=lambda rubro: rubro[1])

    def generar_informe(self) -> str:
        # Arma el reporte completo como texto, para poder imprimirlo o compararlo.
        lineas: list[str] = []
        lineas.append(f"ACTIVIDAD: {self.nombre}")
        lineas.append("-" * 48)

        for rubro, costo in self.rubros:
            lineas.append(f"{rubro:<20} ${costo:>12,.2f}")

        lineas.append("-" * 48)
        lineas.append(f"SUBTOTAL:             ${self.calcular_subtotal():>12,.2f}")
        lineas.append(
            f"IMPREVISTOS ({self.porcentaje_imprevistos:>4.1f}%): "
            f"${self.calcular_imprevistos():>12,.2f}"
        )
        lineas.append(f"TOTAL:                ${self.calcular_total():>12,.2f}")
        lineas.append(f"POR PERSONA:          ${self.calcular_costo_por_persona():>12,.2f}")
        lineas.append(f"PRESUPUESTO:          ${self.presupuesto:>12,.2f}")
        lineas.append(self.evaluar_presupuesto())

        rubro_top = self.rubro_mas_costoso()
        if rubro_top is not None:
            nombre_top, costo_top = rubro_top
            lineas.append(f"RUBRO MÁS COSTOSO:    {nombre_top} (${costo_top:,.2f})")

        return "\n".join(lineas)

## 2. Creación de dos actividades

Se crean dos objetos `Actividad` con presupuestos, asistentes y porcentajes de imprevistos distintos, cada uno con su propia lista de rubros. Al ser objetos independientes, no comparten estado entre sí.

In [ ]:
actividad_1: Actividad = Actividad(
    nombre="Muestra de proyectos",
    presupuesto=350_000.0,
    asistentes=80,
    porcentaje_imprevistos=10.0,
)
actividad_1.agregar_rubro("Espacio", 120_000.0)
actividad_1.agregar_rubro("Técnica", 95_000.0)
actividad_1.agregar_rubro("Comunicación", 38_000.0)
actividad_1.agregar_rubro("Catering", 72_000.0)

actividad_2: Actividad = Actividad(
    nombre="Jornada de puertas abiertas",
    presupuesto=500_000.0,
    asistentes=200,
    porcentaje_imprevistos=15.0,
)
actividad_2.agregar_rubro("Espacio", 90_000.0)
actividad_2.agregar_rubro("Técnica", 60_000.0)
actividad_2.agregar_rubro("Comunicación", 55_000.0)
actividad_2.agregar_rubro("Catering", 180_000.0)
actividad_2.agregar_rubro("Seguridad", 40_000.0)

actividades: list[Actividad] = [actividad_1, actividad_2]

print(f"Objetos creados: {[actividad.nombre for actividad in actividades]}")

## 3. Informe individual

Como `generar_informe()` vive en la clase, mostrar el reporte de cualquier actividad es siempre la misma línea de código, sin importar cuántos rubros o actividades haya.

In [ ]:
for actividad in actividades:
    print(actividad.generar_informe())
    print()

## 4. Comparación entre actividades

`comparar_actividades()` es una función aparte (no un método) porque compara *varios* objetos entre sí, no describe a uno solo. Está armada para recibir cualquier cantidad de actividades: sirve para comparar 2, 5 o 20 sin cambiar el código.

In [ ]:
def comparar_actividades(*actividades: Actividad) -> None:
    if not actividades:
        print("No hay actividades para comparar.")
        return

    encabezado: str = (
        f"{'ACTIVIDAD':<28}{'TOTAL':>14}{'POR PERSONA':>16}{'ESTADO':>14}"
    )
    print(encabezado)
    print("-" * len(encabezado))

    for actividad in actividades:
        total: float = actividad.calcular_total()
        por_persona: float = actividad.calcular_costo_por_persona()
        estado: str = "Viable" if actividad.presupuesto >= total else "No viable"

        print(
            f"{actividad.nombre:<28}"
            f"${total:>12,.2f}"
            f"${por_persona:>14,.2f}"
            f"{estado:>14}"
        )

    actividad_mas_cara: Actividad = max(actividades, key=lambda actividad: actividad.calcular_total())
    print("-" * len(encabezado))
    print(f"Actividad con mayor costo total: {actividad_mas_cara.nombre}")


comparar_actividades(*actividades)

## Conclusiones

- El **rubro más costoso** se calcula con `max()` sobre la lista de tuplas, usando el costo como criterio de orden (`key=lambda rubro: rubro[1]`).
- Los **imprevistos** se resolvieron como un porcentaje del subtotal, guardado como atributo del objeto: cada actividad puede definir el suyo propio.
- La **comparación** se resolvió con una función que recibe `*actividades` (número variable de argumentos) en lugar de asumir que siempre van a ser dos. Así el mismo código sirve si mañana se agregan más actividades a la lista.